# 10 — Krahasimi i modeleve dhe pasiguria

        **Pyetja udhëheqëse:** Kur disa modele funksionojnë, si vendosim cilin të besojmë?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të krahasohen 5 familje klasifikuesish në të njëjtin problem.
- Të përdoret i njëjti protokoll validimi.
- Të maten kohët, stabiliteti, kalibrimi dhe robustësia.
- Të diskutohet pasiguria pa 'fitues absolut'.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("10 — Krahasimi i modeleve dhe pasiguria")
print("Seed:", RANDOM_STATE)

# 1. Dataset i përbashkët: sinjal/sfond — CORE

In [ ]:
n=1400
X0=rng.normal([0,0,0],[1.0,1.1,0.8],size=(900,3))
X1=rng.normal([1.25,1.0,0.8],[0.95,0.9,0.85],size=(500,3))
X=np.vstack([X0,X1]); y=np.r_[np.zeros(len(X0)),np.ones(len(X1))]
df=pd.DataFrame(X,columns=["amplitude","width","skew"])
df["target"]=y
df.head()

In [ ]:
fig,ax=plt.subplots()
ax.scatter(X[y==0,0],X[y==0,1],s=8,label="sfond")
ax.scatter(X[y==1,0],X[y==1,1],s=8,label="sinjal")
ax.set_xlabel("amplitude"); ax.set_ylabel("width"); ax.legend()
ax.set_title("Dataset i përbashkët për krahasim")
plt.show()

# 2. Protokoll i përbashkët — CORE

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.25,stratify=y,random_state=RANDOM_STATE)
models={
    "LogReg": make_pipeline(StandardScaler(),LogisticRegression(max_iter=3000)),
    "kNN": make_pipeline(StandardScaler(),KNeighborsClassifier(11)),
    "Tree": DecisionTreeClassifier(max_depth=5,random_state=RANDOM_STATE),
    "RF": RandomForestClassifier(n_estimators=250,random_state=RANDOM_STATE),
    "SVM": make_pipeline(StandardScaler(),SVC(C=2,probability=True,random_state=RANDOM_STATE))
}

# 3. Validim i kryqëzuar me disa metrika — CORE

In [ ]:
cv=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE)
rows=[]
for name,m in models.items():
    r=cross_validate(m,Xtr,ytr,cv=cv,scoring=["accuracy","precision","recall","f1","roc_auc"],return_train_score=True)
    rows.append({
        "model":name,
        "acc":r["test_accuracy"].mean(),
        "precision":r["test_precision"].mean(),
        "recall":r["test_recall"].mean(),
        "f1":r["test_f1"].mean(),
        "auc":r["test_roc_auc"].mean(),
        "auc_std":r["test_roc_auc"].std(),
        "fit_time":r["fit_time"].mean()
    })
results=pd.DataFrame(rows).set_index("model")
results

In [ ]:
fig,ax=plt.subplots()
results[["f1","auc"]].plot(kind="bar",ax=ax)
ax.set_ylim(0,1); ax.set_ylabel("Score")
ax.set_title("Krahasimi i modeleve")
plt.show()

# 4. Shpërndarja e score-ve ndër fold-e — CORE

In [ ]:
auc_samples={}
for name,m in models.items():
    r=cross_validate(m,Xtr,ytr,cv=cv,scoring="roc_auc")
    auc_samples[name]=r["test_score"]
fig,ax=plt.subplots()
ax.boxplot([auc_samples[k] for k in auc_samples],tick_labels=list(auc_samples))
ax.set_ylabel("ROC-AUC"); ax.set_title("Stabiliteti ndër fold-e")
plt.show()

# 5. Test final i paprekur — CORE

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score
test_rows=[]
for name,m in models.items():
    m.fit(Xtr,ytr)
    pred=m.predict(Xte)
    proba=m.predict_proba(Xte)[:,1]
    test_rows.append((name,f1_score(yte,pred),roc_auc_score(yte,proba)))
pd.DataFrame(test_rows,columns=["model","F1_test","AUC_test"]).set_index("model")

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 6. Sensitivitet ndaj zhurmës — EXTENDED

In [ ]:
noise_levels=np.linspace(0,1.5,8)
curves={name:[] for name in models}
for sig in noise_levels:
    Xnoisy=Xte+rng.normal(0,sig,Xte.shape)
    for name,m in models.items():
        curves[name].append(m.score(Xnoisy,yte))
fig,ax=plt.subplots()
for name,vals in curves.items():
    ax.plot(noise_levels,vals,marker="o",label=name)
ax.set_xlabel("σ e zhurmës së shtuar"); ax.set_ylabel("Accuracy")
ax.legend(); ax.set_title("Robustësia ndaj perturbimit")
plt.show()

# 7. Bootstrap i AUC — EXTENDED

In [ ]:
from sklearn.utils import resample
rf=models["RF"].fit(Xtr,ytr)
p=rf.predict_proba(Xte)[:,1]
boot=[]
for _ in range(400):
    idx=rng.integers(0,len(yte),len(yte))
    if len(np.unique(yte[idx]))<2: continue
    boot.append(roc_auc_score(yte[idx],p[idx]))
fig,ax=plt.subplots()
ax.hist(boot,bins=25)
ax.set_xlabel("AUC bootstrap"); ax.set_title("Pasiguria e AUC për Random Forest")
plt.show()

# 8. Learning curves — EXTENDED

In [ ]:
from sklearn.model_selection import learning_curve
sizes,tr,va=learning_curve(models["LogReg"],Xtr,ytr,cv=cv,scoring="roc_auc",
                            train_sizes=np.linspace(0.15,1.0,6))
fig,ax=plt.subplots()
ax.plot(sizes,tr.mean(axis=1),marker="o",label="trajnim")
ax.plot(sizes,va.mean(axis=1),marker="o",label="validim")
ax.set_xlabel("Madhësia e trajnimit"); ax.set_ylabel("ROC-AUC")
ax.legend(); ax.set_title("Learning curve")
plt.show()

# 9. Kalibrimi — EXTENDED

In [ ]:
from sklearn.calibration import calibration_curve
lr=models["LogReg"].fit(Xtr,ytr)
p=lr.predict_proba(Xte)[:,1]
frac,meanp=calibration_curve(yte,p,n_bins=10)
fig,ax=plt.subplots()
ax.plot(meanp,frac,marker="o")
ax.plot([0,1],[0,1],linestyle="--")
ax.set_xlabel("Probabiliteti i parashikuar"); ax.set_ylabel("Frekuenca reale")
ax.set_title("Kalibrimi probabilistik")
plt.show()

**Interpretim.** Modeli më i mirë varet nga kriteri: performancë mesatare, stabilitet, kohë, interpretueshmëri, kalibrim dhe robustësi mund të favorizojnë zgjedhje të ndryshme.

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Renditni modelet sipas F1 dhe sipas AUC; a është renditja e njëjtë?
2. Matni kohën e parashikimit.
3. Krahasoni performancën me dhe pa standardizim.
4. Përdorni permutation importance për RF.

        ## B. Ushtrim i plotë
        Përcaktoni një skenar ku false negatives kushtojnë 5 herë më shumë se false positives dhe zgjidhni modelin/pragun sipas kësaj kostoje.

        ## C. Playground
        Ndryshoni pabalancimin e klasave, zhurmën, metrikën kryesore dhe parametrat.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.